# 01 — Product Selection and Cleaning

In this notebook, I use the Amazon Reviews 2023 Electronics dataset to prepare the product catalog for ElectroLens. I select up to 1,500 products from each of the 13 categories, keeping products with valid images, prices, ASINs, and at least 20 ratings.

In [1]:
%pip -q install pandas pyarrow tqdm requests

In [2]:
import re
import json
import math
import random
import requests
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm

SEED = 42
MIN_RATINGS = 20
MAX_PER_CATEGORY = 1500

BASE = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw"
META_URL = f"{BASE}/meta_categories/meta_Electronics.jsonl"


In [3]:
def stream_jsonl(url, retries=5):
    done = 0
    for attempt in range(retries):
        try:
            with requests.get(url, stream=True, timeout=60) as response:
                response.raise_for_status()
                for i, line in enumerate(response.iter_lines()):
                    if i < done:
                        continue
                    done = i + 1
                    if line:
                        yield json.loads(line)
            return
        except (requests.exceptions.RequestException, requests.exceptions.ChunkedEncodingError) as exc:
            print(f"Retry {attempt + 1}/{retries} after {done:,} lines: {exc}")
    raise RuntimeError("Metadata download failed")


def pick_image(images):
    if not isinstance(images, list):
        return None
    ordered = sorted(
        (im for im in images if isinstance(im, dict)),
        key=lambda im: im.get("variant") != "MAIN"
    )
    for im in ordered:
        for field in ("large", "hi_res", "thumb"):
            url = im.get(field)
            if isinstance(url, str) and url.startswith(("https://", "http://")):
                return url
    return None


def to_price(value):
    try:
        price = float(value)
        return price if math.isfinite(price) and price > 0 else None
    except (ValueError, TypeError):
        return None


def join_text(value):
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        return " ".join(str(x).strip() for x in value if x is not None)
    return ""


def get_categories(categories):
    if not isinstance(categories, list):
        return None
    parts = [x.strip() for x in categories if isinstance(x, str) and x.strip()]
    if len(parts) < 3 or parts[0] != "Electronics":
        return None
    return {
        "main_group": parts[1],
        "subcategory": parts[-1],
        "cat_path": " > ".join(parts),
    }


In [5]:
sample = []

for product in stream_jsonl(META_URL):
    sample.append(product)
    if len(sample) == 5:
        break

df_sample = pd.DataFrame(sample)
display(df_sample)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Fat Shark,"[Electronics, Television & Video, Video Glasses]","{'Date First Available': 'August 2, 2014', 'Ma...",B00MCW7G9M,None
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],SIIG,"[Electronics, Television & Video, Accessories,...",{'Product Dimensions': '0.83 x 4.17 x 2.05 inc...,B00YT6XQSE,None
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'AL 2Sides Video', 'url': 'https://...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{'Brand': 'Digi-Tatoo', 'Color': 'Fresh Marble...",B07SM135LS,None
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{'Date First Available': 'May 29, 2020', 'Manu...",B089CNGZCW,None
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Verizon,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '11.6 x 6.9 x 3.1 inche...,B004E2Z88O,None


In [6]:
sample = []

for product in stream_jsonl(META_URL):
    sample.append(product)
    if len(sample) == 10000:
        break

df_sample = pd.DataFrame(sample)

In [7]:
pd.set_option('display.max_rows',None)

In [8]:
category_counts = df_sample["categories"].explode().value_counts()

display(category_counts.to_frame("count"))

,count
categories,
Electronics,9263
Computers & Accessories,4211
Camera & Photo,1484
Accessories,1190
"Bags, Cases & Sleeves",1162
Computer Accessories & Peripherals,1120
Laptop Accessories,942
Cases,903
Tablet Accessories,896


In [9]:
category_paths = (
    df_sample["categories"]
    .apply(lambda x: " > ".join(x) if isinstance(x, list) else "Unknown")
    .value_counts()
)

display(category_paths.head(50).to_frame("count"))

print("Unique category paths:", len(category_paths))

,count
categories,
,724
"Electronics > Computers & Accessories > Tablet Accessories > Bags, Cases & Sleeves > Cases",618
"Electronics > Headphones, Earbuds & Accessories > Headphones & Earbuds > Earbud Headphones",242
"Electronics > Headphones, Earbuds & Accessories > Cases",194
Electronics > Computers & Accessories > Computer Accessories & Peripherals > Cables & Accessories > Cables & Interconnects > USB Cables,184
Electronics > Camera & Photo > Lighting & Studio > Photo Studio > Backgrounds,170
Electronics > Computers & Accessories > Computers & Tablets > Laptops > Traditional Laptops,166
Electronics > Television & Video > Accessories > Remote Controls,154
"Electronics > Computers & Accessories > Laptop Accessories > Bags, Cases & Sleeves > Backpacks",135


Unique category paths: 658


In [10]:
pd.set_option("display.max_colwidth", 100)

check = df_sample[["title", "categories", "rating_number"]].sample(
    20, random_state=42
).copy()

check["categories"] = check["categories"].apply(
    lambda x: " > ".join(x) if isinstance(x, list) else ""
)

display(check)

,title,categories,rating_number
6252,"Macally Tablet Holder for Mic Stand, iPad Music Stand or Mic Stand Phone Holder with Strong Hold...",Electronics > Computers & Accessories > Tablet Accessories > Mounts,1607
4684,"Todiys Power Button Board with Cable LS-B844P for Dell Inspiron 15"" 3558 5551 5558 5555 5559 Del...",Electronics > Computers & Accessories > Computer Components > Laptop Replacement Parts,10
1731,"ANNSD for iPod Touch 7th Generation Case, iPod Touch 6th / 5th Case，Slim Flexible TPU Cover Hybr...",Electronics > Portable Audio & Video > MP3 & MP4 Player Accessories > Cases,37
4742,MOSISO Compatible with MacBook Air 13 inch Case 2020 2019 2018 Release A2337 M1 A2179 A1932 Reti...,"Electronics > Computers & Accessories > Laptop Accessories > Bags, Cases & Sleeves > Hard Shell ...",54
4521,"OP/TECH USA Fashion Strap - 3/8-Inch, Steel (1611252)","Electronics > Camera & Photo > Bags & Cases > Binocular, Camera & Camcorder Straps",158
6340,"Case for Samsung Galaxy Tab S2 9.7 inch (SM-T810 T813 T815 T817 T818 T819), 360 Degree Rotating ...","Electronics > Computers & Accessories > Tablet Accessories > Bags, Cases & Sleeves > Cases",483
576,Mocsicka Neon Birthday Backdrop Laser Glow in The Dark Party Decorations Photo Backdrops Let's G...,Electronics > Camera & Photo > Lighting & Studio > Photo Studio > Backgrounds,117
5202,newshijieCOb L15 1Pc Mini in-Ear Wireless Earbuds Bluetooth 5.0 Earbuds Wireless Earphones Bluet...,"Electronics > Headphones, Earbuds & Accessories > Headphones & Earbuds > Earbud Headphones",4
6363,LB1 High Performance Battery for Apple MB985X/A MB985ZP/A [Mid 2009 to Mid 2010] Macbook Pro 15 ...,Electronics > Computers & Accessories > Laptop Accessories > Batteries,9
439,yueton Pack of 10 Wire Lead Battery Storage Box Case Holder for 18650 Button Top Single Battery,"Electronics > Household Batteries, Chargers & Accessories > Battery Holders",325


In [17]:
MAIN_CATEGORIES = [
    "headphones", "laptop", "desktop", "tablet", "monitor", "tv",
    "smartwatch", "router", "camera", "keyboard", "mouse", "speaker",
]
ALL_CATEGORIES = MAIN_CATEGORIES + ["accessories"]

VALID_DEPARTMENTS = {
    "Computers & Accessories", "Camera & Photo",
    "Headphones, Earbuds & Accessories", "Television & Video",
    "Home Audio", "Portable Audio & Video", "Accessories & Supplies",
    "Power Accessories", "Wearable Technology", "Video Projectors",
    "Security & Surveillance", "Household Batteries, Chargers & Accessories",
    "eBook Readers & Accessories", "Car & Vehicle Electronics",
    "GPS, Finders & Accessories",
}

PRODUCT_RULES = {
    "headphones": r"earbud headphones|over.ear headphones|on.ear headphones|in.ear headphones|^headphones$|^earbuds$",
    "laptop": r"traditional laptops|gaming laptops|chromebooks|2.in.1 laptops|^laptops$",
    "desktop": r"desktop computers|all.in.one computers|mini pcs|^desktops$|^towers$",
    "tablet": r"^tablets$|tablet computers",
    "monitor": r"^monitors$|computer monitors",
    "tv": r"led & lcd tvs|oled tvs|qled tvs|smart tvs|^televisions$|^tvs$",
    "smartwatch": r"^smartwatches$|fitness trackers|activity trackers",
    "router": r"^routers$|wireless routers|mesh wi.fi systems",
    "camera": r"digital cameras|mirrorless cameras|dslr cameras|point & shoot digital cameras|webcams|camcorders|digital slr cameras",
    "keyboard": r"^keyboards$|gaming keyboards",
    "mouse": r"^mice$|gaming mice",
    "speaker": r"bluetooth speakers|portable speakers|bookshelf speakers|^speakers$|soundbars",
}
COMPILED_RULES = {k: re.compile(v, re.I) for k, v in PRODUCT_RULES.items()}

ACCESSORY_RULE = re.compile(
    r"cases|covers|sleeves|bags|backpacks|cables|adapters|chargers|"
    r"batteries|mounts|stands|screen protectors|earpads|straps|"
    r"bands|remote controls|decals|replacement parts|tripods|"
    r"connectors|power cords|docking stations|holders|skins|"
    r"accessories|parts|lens caps|cleaning kits",
    re.I,
)


def classify_product(product):
    path = product.get("cat_path", "")

    categories = [
        part.strip()
        for part in path.split(" > ")
        if part.strip()
    ]

    if len(categories) < 3:
        return None

    if categories[0] != "Electronics":
        return None

    if categories[1] not in VALID_DEPARTMENTS:
        return None

    specific = categories[2:]
    leaf = specific[-1]

    for group, pattern in COMPILED_RULES.items():
        if pattern.search(leaf):
            return group

    if any(ACCESSORY_RULE.search(part) for part in specific):
        return "accessories"

    for group, pattern in COMPILED_RULES.items():
        if any(pattern.search(part) for part in specific[:-1]):
            return group

    return None

In [12]:
unique = {}

for row in tqdm(stream_jsonl(META_URL), total=1_600_000, desc="Electronics metadata"):
    title = str(row.get("title") or "").strip()
    asin = row.get("parent_asin")
    cat = get_categories(row.get("categories"))

    if len(title) < 10 or cat is None:
        continue

    if not isinstance(asin, str) or not re.fullmatch(r"[A-Z0-9]{10}", asin):
        continue

    image = pick_image(row.get("images"))
    price = to_price(row.get("price"))

    if image is None or price is None:
        continue

    try:
        rating_number = int(row.get("rating_number") or 0)
        average_rating = float(row.get("average_rating"))
    except (TypeError, ValueError):
        continue

    if rating_number < MIN_RATINGS or not math.isfinite(average_rating) or not 1 <= average_rating <= 5:
        continue

    product = {
        "parent_asin": asin,
        "title": title,
        **cat,
        "store": row.get("store"),
        "price": price,
        "average_rating": average_rating,
        "rating_number": rating_number,
        "features": join_text(row.get("features")),
        "description": join_text(row.get("description")),
        "image_url": image,
    }

    if asin not in unique or rating_number > unique[asin]["rating_number"]:
        unique[asin] = product

print(f"Eligible unique products: {len(unique):,}")

Electronics metadata:   0%|          | 0/1600000 [00:00<?, ?it/s]

Eligible unique products: 254,572


In [28]:
grouped = defaultdict(list)

for item in unique.values():
    group = classify_product(item)

    if group is not None:
        grouped[group].append({
            **item,
            "product_group": group
        })

rng = random.Random(SEED)
selected = []

def sample_popular_random(rows, limit):
    rows = sorted(
        rows,
        key=lambda p: (-p["rating_number"], p["parent_asin"])
    )

    count = min(limit, len(rows))
    top_count = count // 2

    popular = rows[:top_count]
    random_part = rng.sample(
        rows[top_count:],
        count - top_count
    )

    return popular + random_part


for group in MAIN_CATEGORIES:
    selected.extend(
        sample_popular_random(
            grouped[group],
            MAX_PER_CATEGORY
        )
    )


accessory_groups = defaultdict(list)

for product in grouped["accessories"]:
    accessory_groups[product["cat_path"]].append(product)

accessory_selected = []
used_asins = set()

for path in sorted(accessory_groups):
    picks = sample_popular_random(
        accessory_groups[path],
        10
    )

    for product in picks:
        if len(accessory_selected) >= MAX_PER_CATEGORY:
            break

        if product["parent_asin"] not in used_asins:
            accessory_selected.append(product)
            used_asins.add(product["parent_asin"])

    if len(accessory_selected) >= MAX_PER_CATEGORY:
        break


remaining = [
    p for p in grouped["accessories"]
    if p["parent_asin"] not in used_asins
]

slots = MAX_PER_CATEGORY - len(accessory_selected)

if slots > 0:
    accessory_selected.extend(
        sample_popular_random(remaining, slots)
    )

selected.extend(accessory_selected)

products = pd.DataFrame(selected).reset_index(drop=True)

assert products["parent_asin"].is_unique
assert products["product_group"].value_counts().max() <= MAX_PER_CATEGORY

print(f"Final products: {len(products):,}")
print(f"Categories represented: {products['product_group'].nunique()} / {len(ALL_CATEGORIES)}")

display(
    products["product_group"]
    .value_counts()
    .to_frame("count")
)

print("Accessory category paths:",
      products.loc[
          products["product_group"] == "accessories",
          "cat_path"
      ].nunique())

Final products: 18,249
Categories represented: 13 / 13


,count
product_group,
headphones,1500
laptop,1500
desktop,1500
tablet,1500
monitor,1500
smartwatch,1500
camera,1500
mouse,1500
keyboard,1500


Accessory category paths: 166


In [29]:
KW = {
    "headphones": r"headphone|earbud|earphone|headset|airpods|\bbuds\b",
    "laptop": r"laptop|notebook|chromebook|macbook|\bpc\b",
    "desktop": r"desktop|\bpc\b|mini|all-in-one|computer|optiplex|imac|tower",
    "tablet": r"tablet|\bipad\b|\btab\b|fire hd|kindle",
    "monitor": r"monitor|display|screen",
    "tv": r"\btv\b|television|qled|oled",
    "smartwatch": r"watch|fitness|tracker|band|fitbit|garmin",
    "router": r"router|wi-?fi|mesh|extender|modem|hotspot",
    "camera": r"camera|\bcam\b|camcorder|gopro|dslr|mirrorless|webcam",
    "keyboard": r"keyboard|keypad",
    "mouse": r"mouse|mice|trackball",
    "speaker": r"speaker|soundbar|boombox|subwoofer",
}

products["kw_ok"] = [
    bool(re.search(KW[group], str(title), re.I))
    if group in KW else True
    for group, title in zip(products["product_group"], products["title"])
]

print("Overall keyword match:", f"{products['kw_ok'].mean():.1%}")

display(
    products.groupby("product_group")["kw_ok"]
    .agg(["count", "mean"])
    .round(3)
)

sus = products[
    (products["product_group"] != "accessories")
    & (~products["kw_ok"])
]

print("Suspicious products:", len(sus))

if not sus.empty:
    display(
        sus[
            ["title", "product_group", "subcategory", "cat_path"]
        ].sample(min(30, len(sus)), random_state=SEED)
    )

Overall keyword match: 95.2%


,count,mean
product_group,,
accessories,1500,1.000
camera,1500,0.939
desktop,1500,0.955
headphones,1500,0.982
keyboard,1500,0.949
laptop,1500,0.923
monitor,1500,0.979
mouse,1500,0.971
router,1384,0.920


Suspicious products: 885


,title,product_group,subcategory,cat_path
1858,"DELL XPS XPS9350-4007SLV 13.3-Inch Touchscreen (Intel Core i5 6200U 2.30 GHz Processor, 8 GB LPD...",laptop,2 in 1 Laptops,Electronics > Computers & Accessories > Computers & Tablets > Laptops > 2 in 1 Laptops
6967,"Dell P2421, WUXGA 1920 x 1200, 16:9 FHD, Black",monitor,Monitors,Electronics > Computers & Accessories > Monitors
13538,The Gummy Bear Guy (TM) | World's Greatest Gummy (TM) - King Snake | Nearly 2.5 Feet Long! (Cher...,keyboard,Keyboards,"Electronics > Computers & Accessories > Computer Accessories & Peripherals > Keyboards, Mice & A..."
12258,"HyperX PBT Keycaps – Full Key Set, Double Shot PBT Material, English (US) Layout, 104 Keys – White",keyboard,Keyboards,"Electronics > Computers & Accessories > Computer Accessories & Peripherals > Keyboards, Mice & A..."
16713,Audiobank AB-TW220 3.75 inch 600 Watt Car Audio Heavy Duty Compression Super Horn Tweeter/Die ca...,speaker,Tweeters,Electronics > Car & Vehicle Electronics > Car Electronics > Car Audio > Speakers > Tweeters
12389,Microsoft Surface Go Type Cover (Black),keyboard,Keyboards,Electronics > Computers & Accessories > Tablet Accessories > Keyboards
5645,"Panasonic Toughpad FZ-G1 MK1, Intel i5-3437U 1.9GHz, 10.1 WUXGA Multi TouchDigitizer, 128GB SSD,...",tablet,Tablets,Electronics > Computers & Accessories > Computers & Tablets > Tablets
5401,"Microsoft Surface Book 2 (Intel Core i5, 8GB RAM, 256GB) - 13.5in (Renewed)",tablet,Tablets,Electronics > Computers & Accessories > Computers & Tablets > Tablets
2822,"Lenovo Thinkpad X1 Carbon 20A70037US Touch 14-Inch Touchscreen Ultrabook - Core i7-4600U, 14"" Mu...",laptop,Traditional Laptops,Electronics > Computers & Accessories > Computers & Tablets > Laptops > Traditional Laptops
16738,Alpine Electronics PSS-22WRA Weather Resistant Direct Fit Sound System for 2007-2018 Jeep Wrangl...,speaker,Component Speakers,Electronics > Car & Vehicle Electronics > Car Electronics > Car Audio > Speakers > Component Spe...


In [30]:
print("Duplicate ASINs:", products["parent_asin"].duplicated().sum())
print("Missing image URLs:", products["image_url"].isna().sum())
print("Missing prices:", products["price"].isna().sum())

for group in ALL_CATEGORIES:
    sample = products[products["product_group"] == group]
    if sample.empty:
        print(f"{group}: no eligible products")
        continue
    print(f"\n{group.upper()} ({len(sample):,})")
    display(sample[["title", "subcategory", "cat_path"]].sample(
        min(3, len(sample)), random_state=SEED
    ))


Duplicate ASINs: 0
Missing image URLs: 0
Missing prices: 0

HEADPHONES (1,500)


,title,subcategory,cat_path
1116,Kids Headphones Wired Over Ear Headphone Lightweight Stereo Headset Adjustable Headband Headset ...,Over-Ear Headphones,"Electronics > Headphones, Earbuds & Accessories > Headphones & Earbuds > Over-Ear Headphones"
1368,"iKanzi Bone Conduction Headphones, Open-Ear Headphones, Wireless Bluetooth Headphones (Larger)",Over-Ear Headphones,"Electronics > Headphones, Earbuds & Accessories > Headphones & Earbuds > Over-Ear Headphones"
422,Skullcandy Push Ultra True Wireless In-Ear Earbuds - Electric Yellow,Earbud Headphones,"Electronics > Headphones, Earbuds & Accessories > Headphones & Earbuds > Earbud Headphones"



LAPTOP (1,500)


,title,subcategory,cat_path
2616,"Lenovo 2021 Premium Chromebook Flex 3 11.6"" 2-in-1 Touchscreen HD Laptop Business & Student, Med...",Traditional Laptops,Electronics > Computers & Accessories > Computers & Tablets > Laptops > Traditional Laptops
2868,"HP Newest Flagship 15.6 HD Pavilion Laptop for Business and Student, Intel Pentium Quad-Core Pro...",Traditional Laptops,Electronics > Computers & Accessories > Computers & Tablets > Laptops > Traditional Laptops
1922,"HP 2022 Premium 14-inch HD Thin and Light Laptop, Intel Dual-Core Processor, 32GB RAM, 64GB Stor...",Traditional Laptops,Electronics > Computers & Accessories > Computers & Tablets > Laptops > Traditional Laptops



DESKTOP (1,500)


...

,title,subcategory,cat_path
14865,Logitech 910-005032 M325C Wireless Mouse-Brainstorm Peach,Mice,"Electronics > Computers & Accessories > Computer Accessories & Peripherals > Keyboards, Mice & A..."
15117,"2022 Upgrade Wireless Optical Pocket Pen Mouse with Stylus Function, Promi 2.4G USB Digital Opti...",Mice,"Electronics > Computers & Accessories > Computer Accessories & Peripherals > Keyboards, Mice & A..."
14171,"Cougar gaming CGR-XXNB-MB1 Cougar Bunker Gaming Mouse Bungee, Black with an Orange Cougar Symbol",Mice,"Electronics > Computers & Accessories > Computer Accessories & Peripherals > Keyboards, Mice & A..."



SPEAKER (1,500)


,title,subcategory,cat_path
16365,Magnadyne LS2WP Water-Resistant 3 Inch Ceiling Mount Satellite Speaker | White (Sold as a Pair),Ceiling & In-Wall Speakers,Electronics > Home Audio > Speakers > Ceiling & In-Wall Speakers
16617,Renegade RX42 4-Inch Full Range 2-Way Speakers - Set of 2 (Black),Coaxial Speakers,Electronics > Car & Vehicle Electronics > Car Electronics > Car Audio > Speakers > Coaxial Speakers
15671,"JBL Bar 2.1 - Deep Bass Soundbar with 6.5"" Wireless Subwoofer (2019 Model)",Sound Bars,Electronics > Home Audio > Speakers > Sound Bars



ACCESSORIES (1,500)


,title,subcategory,cat_path
17865,"Dr.meter Professional LED Light Meter, Digital Illuminance Meter with 0-200,000 Measuring Ranges...",Light Meters,Electronics > Camera & Photo > Lighting & Studio > Light Meters & Accessories > Light Meters
18117,"Cobra RAD 380 Laser Radar Detector – Long Range Detection, LaserEye Front and Rear Detection, IV...",Radar Detectors,Electronics > Car & Vehicle Electronics > Vehicle Electronics Accessories > Radar Detectors
17171,"Lynkaye Movie Film Video Clapboard irector's Cut Action Scene Clapper Board,Movie Theme Party De...",Film Processing Equipment,Electronics > Camera & Photo > Accessories > Darkroom Supplies > Film Processing Equipment


In [31]:
products.to_parquet("products.parquet", index=False)
print("Saved products.parquet:", products.shape)


Saved products.parquet: (18249, 14)


In [32]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

SAVE_DIR = Path("/content/drive/MyDrive/ElectroLens")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SAVE_PATH = SAVE_DIR / "products.parquet"

products.to_parquet(SAVE_PATH, index=False)

print(f"Saved successfully: {SAVE_PATH}")
print(f"Products: {len(products):,}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved successfully: /content/drive/MyDrive/ElectroLens/products.parquet
Products: 18,249
